In [3]:
import nibabel as nib
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA



In [6]:
# Check if they located in same space

img1 = nib.load(r"D:/New folder/AD/005_S_10835/dipy_fa.nii.gz")
img2 = nib.load(r"D:/New folder/AD/011_S_6303/dipy_fa.nii.gz")
atlas = nib.load(r"D:/New folder/JHU-ICBM-labels-1mm.nii.gz")

print(img1.shape)
print(img2.shape)
print(atlas.shape)

print(img1.affine)
print(img2.affine)

(182, 218, 182)
(182, 218, 182)
(182, 218, 182)
[[  -1.    0.    0.   90.]
 [   0.    1.    0. -126.]
 [   0.    0.    1.  -72.]
 [   0.    0.    0.    1.]]
[[  -1.    0.    0.   90.]
 [   0.    1.    0. -126.]
 [   0.    0.    1.  -72.]
 [   0.    0.    0.    1.]]


In [7]:
atlas = nib.load(r"D:/New folder/JHU-ICBM-labels-1mm.nii.gz").get_fdata()
labels = np.unique(atlas)
print(labels)
print(len(labels))

[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 34. 35.
 36. 37. 38. 39. 40. 41. 42. 43. 44. 45. 46. 47. 48. 49. 50.]
51


# ROI 

In [9]:
fa = nib.load(r"D:/New folder/AD/005_S_10835/dipy_fa.nii.gz").get_fdata()
atlas = nib.load(r"D:/New folder/JHU-ICBM-labels-1mm.nii.gz").get_fdata()
tree = ET.parse(r"D:/New folder/JHU-labels.xml")

In [10]:
root = tree.getroot()

roi_names = {}

for label in root.iter('label'):
    idx = int(label.attrib['index'])
    roi_names[idx] = label.text

# --------------------------
# Extract features
# --------------------------

rows = []

for roi in np.unique(atlas):

    roi = int(roi)

    if roi == 0:
        continue

    mean_fa = np.mean(fa[atlas == roi])

    rows.append({
        "ROI_ID": roi,
        "ROI_Name": roi_names.get(roi, "Unknown"),
        "Mean_FA": mean_fa
    })

df = pd.DataFrame(rows)

print(df)

    ROI_ID                                           ROI_Name   Mean_FA
0        1                         Middle cerebellar peduncle  0.470573
1        2             Pontine crossing tract (a part of MCP)  0.399755
2        3                            Genu of corpus callosum  0.430659
3        4                            Body of corpus callosum  0.426536
4        5                        Splenium of corpus callosum  0.440622
5        6                 Fornix (column and body of fornix)  0.118447
6        7                              Corticospinal tract R  0.399678
7        8                              Corticospinal tract L  0.408528
8        9                                 Medial lemniscus R  0.511281
9       10                                 Medial lemniscus L  0.489366
10      11                   Inferior cerebellar peduncle R    0.430555
11      12                     Inferior cerebellar peduncle L  0.452715
12      13                     Superior cerebellar peduncle R  0

# ROI Extraction

In [7]:


# =====================================================
# PATHS
# =====================================================

ROOT = r"D:\New folder"

ATLAS_PATH = r"D:\New folder\JHU-ICBM-labels-1mm.nii.gz"

OUTPUT_CSV = r"D:\New folder\DTI_ROI_featuresMC.csv"

# =====================================================
# LOAD ATLAS
# =====================================================

print("Loading atlas...")

atlas_img = nib.load(ATLAS_PATH)
atlas = atlas_img.get_fdata()

roi_ids = sorted(
    [int(x) for x in np.unique(atlas) if x != 0]
)

print(f"Found {len(roi_ids)} ROIs")

# =====================================================
# EXTRACT FEATURES
# =====================================================

rows = []
subject_counter = 1

CLASSES = ["AD", "MCI", "CN"]

for label in CLASSES:

    label_dir = os.path.join(ROOT, label)

    if not os.path.exists(label_dir):
        print(f"Skipping missing class: {label}")
        continue

    subjects = sorted(os.listdir(label_dir))

    for subject in subjects:

        subject_dir = os.path.join(label_dir, subject)

        if not os.path.isdir(subject_dir):
            continue

        print(f"Processing {subject}")

        row = {}

        row["Subject"] = f"S{subject_counter}"
        row["Original_ID"] = subject
        row["Label"] = label

        subject_counter += 1

        # ---------------------------------------------
        # FILES
        # ---------------------------------------------

        file_dict = {
            "FA": "dipy_fa.nii.gz",
            "MD": "dipy_md.nii.gz",
            "AD": "AD.nii.gz",
            "RD": "RD.nii.gz"
        }

        metric_data = {}

        missing_file = False

        for metric, filename in file_dict.items():

            filepath = os.path.join(subject_dir, filename)

            if not os.path.exists(filepath):

                print(f"Missing: {filepath}")
                missing_file = True
                break

            metric_data[metric] = nib.load(
                filepath
            ).get_fdata()

        if missing_file:
            continue

        # ---------------------------------------------
        # ROI FEATURES
        # ---------------------------------------------

        for roi in roi_ids:

            mask = atlas == roi

            for metric in ["FA", "MD", "AD", "RD"]:

                value = np.mean(
                    metric_data[metric][mask]
                )

                feature_name = f"ROI{roi}_{metric}"

                row[feature_name] = value

        rows.append(row)

# =====================================================
# SAVE DATAFRAME
# =====================================================

df = pd.DataFrame(rows)

print("\nDataset shape:")
print(df.shape)

print("\nFirst columns:")
print(df.iloc[:, :10].head())

df.to_csv(OUTPUT_CSV, index=False)

print("\nSaved to:")
print(OUTPUT_CSV)

Loading atlas...
Found 50 ROIs
Processing 005_S_10835
Processing 011_S_6303
Processing 033_S_10027
Processing 033_S_10049
Processing 073_S_10105
Processing 073_S_10105a
Processing 389_S_10676
Processing 389_S_10860
Processing 941_S_10001
Processing 941_S_10085
Processing 941_S_10085a
Processing 002_S_1155
Processing 002_S_4799
Processing 003_S_10426
Processing 003_S_6258
Processing 003_S_6268
Processing 005_S_10205
Processing 005_S_10658
Processing 024_S_2239
Processing 033_S_10025
Processing 035_S_10340
Processing 035_S_6480
Processing 002_S_1280
Processing 002_S_4213
Processing 002_S_6103
Processing 003_S_4350
Processing 003_S_6259
Processing 005_S_10646
Processing 007_S_4488
Processing 020_S_6185
Processing 020_S_6504
Processing 023_S_4164
Processing 941_S_6499

Dataset shape:
(33, 203)

First columns:
  Subject  Original_ID Label   ROI1_FA   ROI1_MD   ROI1_AD   ROI1_RD  \
0      S1  005_S_10835    AD  0.470573  0.000840  0.001267  0.000626   
1      S2   011_S_6303    AD  0.387696 

# Feature Selection Using ANOVA

In [ ]:

# =====================================================
# DATA
# =====================================================

df = pd.read_csv(
    r"D:\New folder\Codes\AD vs CN\DTI_ROI_features.csv"
)

# =====================================================
# SELECT METRIC
# =====================================================
# Change _MD to _FA, _AD, or _RD if needed

metric = "_RD"

feature_cols = [
    c for c in df.columns
    if c.endswith(metric)
]

X = df[feature_cols]
y = df["Label"]

print("Number of features:", X.shape[1])

# =====================================================
# ANOVA
# =====================================================

F_scores, p_values = f_classif(X, y)

anova_df = pd.DataFrame({
    "Feature": feature_cols,
    "F_score": F_scores,
    "p_value": p_values
})

# =====================================================
# ROI NUMBER
# =====================================================

def extract_roi_id(feature_name):
    m = re.search(r"ROI(\d+)_", feature_name)
    return int(m.group(1))

anova_df["ROI_ID"] = anova_df["Feature"].apply(
    extract_roi_id
)

# =====================================================
# LOAD JHU XML
# =====================================================

tree = ET.parse(
    r"D:\New folder\JHU-labels.xml"
)

root = tree.getroot()

roi_dict = {}

for label in root.iter("label"):

    if "index" in label.attrib:

        roi_dict[
            int(label.attrib["index"])
        ] = label.text

# =====================================================
# MAP ROI NAME
# =====================================================

anova_df["ROI_Name"] = anova_df["ROI_ID"].map(
    roi_dict
)

# =====================================================
# SORT
# =====================================================

anova_df = anova_df.sort_values(
    by="F_score",
    ascending=False
)

top5 = anova_df.head(5)

# =====================================================
# OUTPUT
# =====================================================

top5 = top5[
    [
        "ROI_ID",
        "ROI_Name",
        "Feature",
        "F_score",
        "p_value"
    ]
]

print("\nTop 5 ANOVA Features\n")
print(top5.to_string(index=False))
# =========================
# ANOVA
# =========================
F, p = f_classif(X, y)

anova_df = pd.DataFrame({
    "Feature": X.columns,
    "F_score": F,
    "p_value": p
})

anova_df = anova_df.sort_values(
    "F_score",
    ascending=False
)

top5 = anova_df.head(5)

Number of features: 50

Top 5 ANOVA Features

 ROI_ID                                                                                               ROI_Name  Feature   F_score  p_value
     25                                                                              Superior corona radiata R ROI25_RD 16.626508 0.000587
     23                                                                              Anterior corona radiata R ROI23_RD 16.234334 0.000657
     31 Sagittal stratum (include inferior longitidinal fasciculus and inferior fronto-occipital fasciculus) R ROI31_RD 14.057182 0.001263
     26                                                                              Superior corona radiata L ROI26_RD 10.041177 0.004828
     43                  Superior fronto-occipital fasciculus (could be a part of anterior internal capsule) R ROI43_RD  9.573564 0.005721


In [ ]:
# =====================================================
# DATA (CN vs MCI vs AD)
# =====================================================

df = pd.read_csv(
    r"D:\New folder\DTI_ROI_featuresMC.csv"
)

# =====================================================
# SELECT METRIC
# =====================================================
# "_FA", "_MD", "_AD", "_RD"

metric = "_RD"

feature_cols = [
    c for c in df.columns
    if c.endswith(metric)
]

X = df[feature_cols]
y = df["Label"]

print("Number of features:", X.shape[1])

# =====================================================
# ANOVA FEATURE RANKING
# =====================================================

F_scores, p_values = f_classif(X, y)

anova_df = pd.DataFrame({
    "Feature": feature_cols,
    "F_score": F_scores,
    "p_value": p_values
})

# =====================================================
# EXTRACT ROI NUMBER
# =====================================================

def extract_roi_id(feature_name):
    match = re.search(r"ROI(\d+)_", feature_name)
    return int(match.group(1))

anova_df["ROI_ID"] = anova_df["Feature"].apply(
    extract_roi_id
)

# =====================================================
# LOAD JHU ATLAS LABELS
# =====================================================

tree = ET.parse(
    r"D:\New folder\JHU-labels.xml"
)

root = tree.getroot()

roi_dict = {}

for label in root.iter("label"):

    if "index" in label.attrib:

        roi_dict[
            int(label.attrib["index"])
        ] = label.text

# =====================================================
# MAP ROI NAMES
# =====================================================

anova_df["ROI_Name"] = (
    anova_df["ROI_ID"]
    .map(roi_dict)
)

# =====================================================
# SORT BY ANOVA SCORE
# =====================================================

anova_df = anova_df.sort_values(
    by="F_score",
    ascending=False
)

top5 = anova_df.head(5)

# =====================================================
# DISPLAY RESULTS
# =====================================================

top5 = top5[
    [
        "ROI_ID",
        "ROI_Name",
        "Feature",
        "F_score",
        "p_value"
    ]
]

print("\nTop 5 ANOVA Features\n")
print(top5.to_string(index=False))

Number of features: 50

Top 5 ANOVA Features

 ROI_ID                                                                                               ROI_Name  Feature  F_score  p_value
     23                                                                              Anterior corona radiata R ROI23_RD 8.878511 0.000936
     31 Sagittal stratum (include inferior longitidinal fasciculus and inferior fronto-occipital fasciculus) R ROI31_RD 7.229731 0.002738
     25                                                                              Superior corona radiata R ROI25_RD 6.721966 0.003872
     43                  Superior fronto-occipital fasciculus (could be a part of anterior internal capsule) R ROI43_RD 6.090038 0.006028
     45                                                                 Inferior fronto-occipital fasciculus R ROI45_RD 5.940939 0.006705
